In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_silver/13_silver_devices
# ============================================================

import dlt
from pyspark.sql import functions as F

TEMPORAL_VALID_RULE_DATE = "stop_date IS NULL OR stop_date >= start_date"


@dlt.table(name="covid19_lakehouse.silver.devices", comment="Devices with correct types")
@dlt.expect_all_or_drop({
    "valid_patient_id": "patient_id IS NOT NULL",
    "valid_encounter_id": "encounter_id IS NOT NULL",
})
@dlt.expect_or_drop("valid_stop_after_start", TEMPORAL_VALID_RULE_DATE)
def silver_devices():
    df = dlt.read_stream("covid19_lakehouse.bronze.devices")
    return (
        df.select(
            F.col("PATIENT").alias("patient_id"),
            F.col("ENCOUNTER").alias("encounter_id"),
            F.col("CODE").alias("device_code"),
            F.col("DESCRIPTION").alias("device_description"),
            F.col("UDI").alias("udi"),
            F.to_date("START").alias("start_date"),
            F.to_date("STOP").alias("stop_date"),
            F.when(F.col("STOP").isNull(), True).otherwise(False).alias("is_active"),
        )
    )


@dlt.table(name="covid19_lakehouse.quarantine.devices", comment="Devices rejected by Silver DQ rules")
def quarantine_devices():
    df = dlt.read_stream("covid19_lakehouse.bronze.devices")
    return (
        df.filter(
            F.col("PATIENT").isNull()
            | F.col("ENCOUNTER").isNull()
            | (F.col("STOP").isNotNull() & (F.to_date("STOP") < F.to_date("START")))
        )
        .select(
            F.lit("devices").alias("source_table"),
            F.lit("null_fk / stop_before_start").alias("rule_violated"),
            F.col("PATIENT").alias("patient_id"),
            F.col("ENCOUNTER").alias("encounter_id"),
            F.col("START").alias("raw_start"),
            F.col("STOP").alias("raw_stop"),
            F.current_timestamp().alias("quarantined_at"),
        )
    )